## 05b_gravity_network_pl
  Polish counterpart of 05: builds the NUTS3 matrix (73 regions) from the GUS gmina-level
  matrix, then the same models and metrics; flows below 3 are not published by GUS.
  Output: data/processed/05b_gravity_pl/, CZ-PL comparison tables. Environment: nuts3-mobility.

In [ ]:
# PROJ data fix: must run BEFORE geopandas / pyproj / skmob are imported
import os
import sys
from pathlib import Path
from importlib.util import find_spec


def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec and spec.submodule_search_locations:  # pip wheel with bundled data
        pyproj_dir = Path(list(spec.submodule_search_locations)[0])
        candidates.append(pyproj_dir / "proj_dir" / "share" / "proj")
    for c in candidates:
        if (c / "proj.db").exists():
            return c
    return None


proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)
    print("PROJ data: proj.db found, environment variables set")
else:
    print("PROJ data: proj.db NOT found in the expected places")

import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import shapely.wkb
from geopandas.array import GeometryArray
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import skmob
from skmob.models.gravity import Gravity
from skmob.models.radiation import Radiation

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
COUNTRY = "PL"
YEAR = 2023                      # same year as notebooks 04 and 05
CAPITAL_ID = "PL911"             # Miasto Warszawa; section 1 prints the name and asserts the id exists
N_REPS = 20                      # stochastic realizations per model
SEED_BASE = 0                    # realization i uses seed SEED_BASE + i
CRS_GEO = "EPSG:4326"
CRS_METRIC = "EPSG:3035"

# own table columns
COL_ID, COL_POP = "nuts3_id", "population"
COL_COUNTRY, COL_YEAR = "country_code", "year"

# observed flows (NUTS3 level, made in section 0 of this notebook; only pairs with flow >= 3, no diagonal)
FLOWS_CSV = config.INTERIM_DIR / "05b_gravity_pl" / "PL_nuts3_flows.csv"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"

# inputs / outputs (relative paths only)
NUTS3_TABLE = config.PROCESSED_DIR / "nuts3_master_table.parquet"
GISCO_GEOJSON = config.RAW_DIR / "nuts3" / "NUTS_RG_03M_2024_4326.geojson"
CHECKPOINT_DIR = config.PROCESSED_DIR / "05b_gravity_pl"   # realizations + summary tables
CZ_SUMMARY_CSV = config.PROCESSED_DIR / "05_gravity_cz" / "metrics_summary.csv"  # for the CZ-PL comparison
FIG_DIRS = [Path("../figures"), Path("../plots")]
GRAPH_PATH = config.PROCESSED_DIR / "graph_pl_gravity.graphml"
MODELS = ["gravity_power", "gravity_exp", "radiation"]
# ------------------------------------------------------------------

for d in [CHECKPOINT_DIR, *FIG_DIRS]:
    d.mkdir(parents=True, exist_ok=True)

print("Config OK:", COUNTRY, YEAR, f"{N_REPS} realizations/model")

In [ ]:
import requests

# ----------------------------- CONFIG (section 0) -----------------------------
GUS_DIR = config.RAW_DIR / "gus"
GUS_NAME = "macierz_przeplywow_ludnosci_zwiazanych_z_zatrudnieniem.xlsx"
GUS_URL = ("https://stat.gov.pl/download/gfx/portalinformacyjny/pl/defaultaktualnosci/"
           "6536/9/2/1/" + GUS_NAME)
ARCHIVE_PREFIX = "https://web.archive.org/web/2025id_/"   # web archive copy of the GUS file
MIN_BYTES = 1_000_000                                    # a smaller file cannot be the full matrix
ZIP_MAGIC = b"PK"                                        # an xlsx file is a zip archive
TIMEOUT = (20, 300)
# ------------------------------------------------------------------------------

GUS_DIR.mkdir(parents=True, exist_ok=True)
gus_path = GUS_DIR / GUS_NAME
gus_rel = os.path.relpath(gus_path)

if gus_path.exists() and gus_path.stat().st_size > MIN_BYTES:
    print("Already present, skipped:", gus_rel)
else:
    try:
        r = requests.get(ARCHIVE_PREFIX + GUS_URL, timeout=TIMEOUT)
        r.raise_for_status()
        if r.content[:2] != ZIP_MAGIC:
            raise ValueError("not an xlsx (zip header missing)")
        gus_path.write_bytes(r.content)
    except Exception as e:
        print("Download failed:", type(e).__name__)
        print("Manual fix: download the file (GUS or web archive) and put it here:", gus_rel)

if not gus_path.exists():
    raise FileNotFoundError("missing file: " + gus_rel)
print(gus_rel, f"{gus_path.stat().st_size / 1e6:.2f} MB")

In [ ]:
# ----------------------------- CONFIG (section 0.2) -----------------------------
INTERIM_PL_DIR = config.INTERIM_DIR / "05b_gravity_pl"
GMINA_OD_PARQUET = INTERIM_PL_DIR / "gmina_od.parquet"
CHECKS_CSV = CHECKPOINT_DIR / "gus_matrix_checks.csv"
OUT_CMP_CSV = CHECKPOINT_DIR / "gus_control_out.csv"
IN_CMP_CSV = CHECKPOINT_DIR / "gus_control_in.csv"

# original sheet names of the GUS file (not translated)
SHEET_MATRIX = "Macierz przepływów"
SHEET_OUT = "Wyjeżdżający z gmin"
SHEET_IN = "Przyjeżdżający do gmin"
CONTROL_CODE_POS, CONTROL_VALUE_POS = 0, 2    # column positions in the control sheets

CODE_REGEX = r"\d{7}"                         # GUS unit code
P6_LEN = 6                                    # first 6 digits identify the whole gmina
GUS_TOTAL_REFERENCE = 5_182_600               # total commuters published by GUS (5 182.6 thousand)
# ------------------------------------------------------------------------------------

INTERIM_PL_DIR.mkdir(parents=True, exist_ok=True)

if GMINA_OD_PARQUET.exists():
    gmina_od = pd.read_parquet(GMINA_OD_PARQUET)
    print("Loaded from checkpoint:", os.path.relpath(GMINA_OD_PARQUET), gmina_od.shape)
else:
    xl = pd.ExcelFile(gus_path)
    print("Sheets:", xl.sheet_names)
    mat = xl.parse(SHEET_MATRIX, dtype=str)
    RC, RN, WC, WN, FL = list(mat.columns)[:5]     # residence code/name, workplace code/name, flow

    # 1) codes and flows
    good = mat[RC].str.fullmatch(CODE_REGEX, na=False) & mat[WC].str.fullmatch(CODE_REGEX, na=False)
    data = mat[good].copy()
    data["flow_num"] = pd.to_numeric(data[FL], errors="coerce")
    if data["flow_num"].isna().any():
        raise ValueError("non-numeric flow values in the matrix")

    # 2) split into between-gmina and within-gmina (parts of one gmina) rows
    pair = pd.DataFrame({
        COL_FLOW_ORIGIN: data[RC].str[:P6_LEN].to_numpy(),
        COL_FLOW_DEST: data[WC].str[:P6_LEN].to_numpy(),
        COL_FLOW: data["flow_num"].to_numpy(),
    })
    same_gmina = pair[COL_FLOW_ORIGIN] == pair[COL_FLOW_DEST]
    within = pair[same_gmina]
    gmina_od = (pair[~same_gmina]
                .groupby([COL_FLOW_ORIGIN, COL_FLOW_DEST], as_index=False)[COL_FLOW].sum())

    # 3) control sheets: outflow and inflow per gmina
    ctl = {}
    for label, sheet in (("out", SHEET_OUT), ("in", SHEET_IN)):
        s = xl.parse(sheet, dtype=str)
        code_col, val_col = s.columns[CONTROL_CODE_POS], s.columns[CONTROL_VALUE_POS]
        s = s[s[code_col].str.fullmatch(CODE_REGEX, na=False)]
        ctl[label] = (pd.to_numeric(s.set_index(code_col)[val_col])
                      .groupby(lambda k: k[:P6_LEN]).sum())

    def compare(label, key):
        """Control sheet total vs. between-gmina flows + within-gmina flows (>= 3 only)."""
        tbl = pd.concat([ctl[label].rename("control"),
                         gmina_od.groupby(key)[COL_FLOW].sum().rename("between"),
                         within.groupby(key)[COL_FLOW].sum().rename("within_parts")], axis=1).fillna(0)
        tbl["remainder"] = tbl["control"] - tbl["between"] - tbl["within_parts"]
        return tbl

    out_cmp, in_cmp = compare("out", COL_FLOW_ORIGIN), compare("in", COL_FLOW_DEST)

    # 4) checks table
    total = float(data["flow_num"].sum())
    checks = pd.Series({
        "matrix_rows": len(mat),
        "rows_with_invalid_code": int((~good).sum()),
        "flow_not_positive_integer": int(((data["flow_num"] % 1 != 0) | (data["flow_num"] <= 0)).sum()),
        "duplicated_residence_work_pairs": int(data.duplicated([RC, WC]).sum()),
        "min_flow": int(data["flow_num"].min()),
        "total_flow": int(total),
        "total_minus_gus_reference": int(total - GUS_TOTAL_REFERENCE),
        "gmina_pairs": len(gmina_od),
        "gmina_pairs_total_flow": int(gmina_od[COL_FLOW].sum()),
        "within_gmina_total_flow": int(within[COL_FLOW].sum()),
        "control_out_negative_remainder": int((out_cmp["remainder"] < 0).sum()),
        "control_in_negative_remainder": int((in_cmp["remainder"] < 0).sum()),
        "control_out_remainder_sum": int(out_cmp["remainder"].sum()),
        "control_in_remainder_sum": int(in_cmp["remainder"].sum()),
    }, name="value")
    checks.to_csv(CHECKS_CSV, header=True)
    out_cmp.to_csv(OUT_CMP_CSV)
    in_cmp.to_csv(IN_CMP_CSV)
    gmina_od.to_parquet(GMINA_OD_PARQUET, index=False)

    print(checks.to_string())
    print("Saved:", os.path.relpath(GMINA_OD_PARQUET))
    print("Saved:", os.path.relpath(CHECKS_CSV), os.path.relpath(OUT_CMP_CSV), os.path.relpath(IN_CMP_CSV))

In [ ]:
# ----------------------------- CONFIG (section 0.3) -----------------------------
EUROSTAT_DIR = config.RAW_DIR / "eurostat"
LAU_NUTS_NAME = "EU-27-LAU-2021-NUTS-2021.xlsx"
LAU_NUTS_URL = "https://ec.europa.eu/eurostat/documents/345175/501971/" + LAU_NUTS_NAME
LAU_NUTS_MIN_BYTES = 100_000                    # a smaller file cannot be the full table
SHEET_LAU = COUNTRY                             # the sheet is named by the country code
COL_NUTS_SRC, COL_LAU_SRC = "NUTS 3 CODE", "LAU CODE"   # original column names of the Eurostat sheet
LAU_ID_LEN = 13
P6_SLICES = ((4, 6), (9, 13))                   # p6 = LAU code [4:6] + [9:13]
MAPPING_CSV = INTERIM_PL_DIR / "gmina_nuts3.csv"
MAP_COLS = {"p6": "p6", "lau": "lau_code", "nuts3": "nuts3_code"}
MAX_SHOW = 10
# --------------------------------------------------------------------------------

if MAPPING_CSV.exists():
    lau_nuts = pd.read_csv(MAPPING_CSV, dtype=str)
    print("Loaded from checkpoint:", os.path.relpath(MAPPING_CSV), lau_nuts.shape)
else:
    EUROSTAT_DIR.mkdir(parents=True, exist_ok=True)
    lau_nuts_path = EUROSTAT_DIR / LAU_NUTS_NAME
    lau_nuts_rel = os.path.relpath(lau_nuts_path)

    if lau_nuts_path.exists() and lau_nuts_path.stat().st_size > LAU_NUTS_MIN_BYTES:
        print("Already present, skipped:", lau_nuts_rel)
    else:
        try:
            r = requests.get(LAU_NUTS_URL, timeout=TIMEOUT)
            r.raise_for_status()
            if r.content[:2] != ZIP_MAGIC:
                raise ValueError("not an xlsx (zip header missing)")
            lau_nuts_path.write_bytes(r.content)
        except Exception as e:
            print("Download failed:", type(e).__name__)
            print("Manual fix: download the Eurostat 'LAU - NUTS 2021' table and put it here:", lau_nuts_rel)
    if not lau_nuts_path.exists():
        raise FileNotFoundError("missing file: " + lau_nuts_rel)

    src = (pd.read_excel(lau_nuts_path, sheet_name=SHEET_LAU, dtype=str)[[COL_LAU_SRC, COL_NUTS_SRC]]
           .astype(str).apply(lambda s: s.str.strip()))
    if not src[COL_LAU_SRC].str.fullmatch(rf"\d{{{LAU_ID_LEN}}}").all():
        raise ValueError("not every LAU code is a 13-digit string")
    if src[COL_LAU_SRC].duplicated().any():
        raise ValueError("duplicated LAU code in the Eurostat sheet")

    lau = src[COL_LAU_SRC]
    lau_nuts = pd.DataFrame({
        MAP_COLS["p6"]: lau.str[slice(*P6_SLICES[0])] + lau.str[slice(*P6_SLICES[1])],
        MAP_COLS["lau"]: lau,
        MAP_COLS["nuts3"]: src[COL_NUTS_SRC],
    })
    lau_nuts.to_csv(MAPPING_CSV, index=False)
    print("Saved:", os.path.relpath(MAPPING_CSV), lau_nuts.shape)

# Checks (always run)
p6_col, nuts_col = MAP_COLS["p6"], MAP_COLS["nuts3"]
dup_p6 = int(lau_nuts[p6_col].duplicated().sum())

used_codes = set(gmina_od[COL_FLOW_ORIGIN]) | set(gmina_od[COL_FLOW_DEST])
unmapped = sorted(used_codes - set(lau_nuts[p6_col]))
n_without_flow = len(set(lau_nuts[p6_col]) - used_codes)

mt = pd.read_parquet(NUTS3_TABLE, columns=[COL_ID, COL_COUNTRY, COL_YEAR])
master_ids = set(mt.loc[(mt[COL_COUNTRY] == COUNTRY) & (mt[COL_YEAR] == YEAR), COL_ID])
map_ids = set(lau_nuts[nuts_col])
only_map, only_master = sorted(map_ids - master_ids), sorted(master_ids - map_ids)

print("Mapping rows:", len(lau_nuts), "| NUTS3 units in mapping:", len(map_ids),
      "| NUTS3 units in master table:", len(master_ids))
print("Duplicated p6 keys:", dup_p6)
print("Codes in the gmina table without mapping:", len(unmapped), unmapped[:MAX_SHOW])
print("Mapped gminas without any flow:", n_without_flow)
print("NUTS3 only in mapping:", only_map[:MAX_SHOW], "| only in master table:", only_master[:MAX_SHOW])
if dup_p6 or unmapped or only_map or only_master:
    raise ValueError("the gmina -> NUTS3 mapping does not match the data, see the lines above")

In [ ]:
# Between-region flows: map every gmina pair to a NUTS3 pair and sum (section 0.4)
unit_of = dict(zip(lau_nuts[MAP_COLS["p6"]], lau_nuts[MAP_COLS["nuts3"]]))

pairs = pd.DataFrame({
    COL_FLOW_ORIGIN: gmina_od[COL_FLOW_ORIGIN].map(unit_of),
    COL_FLOW_DEST: gmina_od[COL_FLOW_DEST].map(unit_of),
    COL_FLOW: gmina_od[COL_FLOW],
})
if pairs.isna().any().any():
    raise ValueError("a gmina code could not be mapped to a NUTS3 unit")

same_unit = pairs[COL_FLOW_ORIGIN] == pairs[COL_FLOW_DEST]
flows_new = (pairs[~same_unit]
             .groupby([COL_FLOW_ORIGIN, COL_FLOW_DEST], as_index=False)[COL_FLOW].sum())
flows_new.to_csv(FLOWS_CSV, index=False)

# Checks
total_gmina = float(gmina_od[COL_FLOW].sum())
total_new = float(flows_new[COL_FLOW].sum())
total_within = float(pairs.loc[same_unit, COL_FLOW].sum())
n_units = flows_new[COL_FLOW_ORIGIN].nunique()
print("NUTS3 pairs:", len(flows_new), "| units as origin:", n_units,
      "| share of possible pairs:", round(len(flows_new) / (n_units * (n_units - 1)), 3))
print("Total flow (between NUTS3):", int(total_new), "| within the same NUTS3:", int(total_within),
      "| gmina-pair total:", int(total_gmina))
print("Total check (between + within - gmina total):", int(total_new + total_within - total_gmina))
print("Smallest NUTS3 flow:", int(flows_new[COL_FLOW].min()))
assert np.isclose(total_new + total_within, total_gmina), "the totals do not add up"
assert flows_new[COL_FLOW].min() >= 3, "a NUTS3 flow is smaller than the gmina-pair cut"
print("Saved:", os.path.relpath(FLOWS_CSV))

In [ ]:
# Load the master table; geometry is stored as WKB bytes
nuts3 = pd.read_parquet(NUTS3_TABLE)

# Element-wise geometry assignment (vectorized assignment fails with this shapely version)
geoms = nuts3["geometry"].apply(
    lambda x: shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x
).tolist()
geom_arr = np.empty(len(geoms), dtype=object)
for i, g in enumerate(geoms):
    geom_arr[i] = g

nuts3 = nuts3.drop(columns="geometry")
nuts3 = gpd.GeoDataFrame(nuts3, geometry=GeometryArray(geom_arr, crs=CRS_GEO))

# Select the country and year
pl = (nuts3[(nuts3[COL_COUNTRY] == COUNTRY) & (nuts3[COL_YEAR] == YEAR)]
      .copy().reset_index(drop=True))

# Centroids and pairwise distances in a metric projection
pl_metric = pl.to_crs(CRS_METRIC)
coords = np.array([[p.x, p.y] for p in pl_metric.geometry.centroid])
dist_matrix_km = squareform(pdist(coords)) / 1000   # metres -> km

ids = pl[COL_ID].tolist()
n_regions = len(ids)

# Region names from the GISCO file (properties only), used for checking and labels
GISCO_COL_ID, GISCO_COL_NAME, GISCO_COL_LEVEL = "NUTS_ID", "NUTS_NAME", "LEVL_CODE"
NUTS_LEVEL = 3
with open(GISCO_GEOJSON, encoding="utf-8") as f:
    gisco_props = pd.DataFrame([ft["properties"] for ft in json.load(f)["features"]])
gisco_country = gisco_props[
    gisco_props[GISCO_COL_ID].str.startswith(COUNTRY)
    & (gisco_props[GISCO_COL_LEVEL] == NUTS_LEVEL)
]
code_to_name = dict(zip(gisco_country[GISCO_COL_ID], gisco_country[GISCO_COL_NAME]))
pl["name"] = pl[COL_ID].map(code_to_name)

# Checks: capital in the regions, flow file ids in the regions
assert CAPITAL_ID in set(ids), "CAPITAL_ID not found among the regions"
flows_raw = pd.read_csv(FLOWS_CSV)
flow_ids = set(flows_raw[COL_FLOW_ORIGIN]) | set(flows_raw[COL_FLOW_DEST])
ids_missing_in_regions = sorted(flow_ids - set(ids))
capital_pop_share = pl.loc[pl[COL_ID] == CAPITAL_ID, COL_POP].iloc[0] / pl[COL_POP].sum()

print("Regions:", n_regions)
print("Distance matrix:", dist_matrix_km.shape)
print("Capital:", CAPITAL_ID, "| name:", code_to_name.get(CAPITAL_ID),
      "| population share:", round(capital_pop_share, 3))
print("Flow-file ids missing among the regions:", ids_missing_in_regions)
print("Regions without a GISCO name:", int(pl["name"].isna().sum()))
print("Missing population values:", int(pl[COL_POP].isna().sum()))
print()
print(pl.nlargest(5, COL_POP)[[COL_ID, "name", COL_POP]].to_string(index=False))

In [ ]:
# Local config
OD_CSV = CHECKPOINT_DIR / "od_observed.csv"

# Long table -> matrix in the region order of pl; missing pairs are 0
dup_pairs = int(flows_raw.duplicated(subset=[COL_FLOW_ORIGIN, COL_FLOW_DEST]).sum())
od = (flows_raw
      .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
      .reindex(index=ids, columns=ids)
      .fillna(0))
observed_offdiag = od.to_numpy(dtype=float).copy()

# Checks
diag_sum = float(np.trace(observed_offdiag))
total_flow = float(observed_offdiag.sum())
tot_outflows = pd.Series(observed_offdiag.sum(axis=1), index=ids)
assert dup_pairs == 0, "duplicated origin-destination pairs"
assert diag_sum == 0, "the diagonal is not empty"
assert np.isclose(total_flow, flows_raw[COL_FLOW].sum()), "total changed during pivot"
assert (tot_outflows > 0).all(), "a region has zero outflow"

od.to_csv(OD_CSV)

pos_share = (observed_offdiag > 0).sum() / (n_regions * (n_regions - 1))
print("OD matrix shape:", od.shape)
print("Duplicated pairs:", dup_pairs, "| diagonal sum:", int(diag_sum))
print("Positive pairs share:", round(float(pos_share), 3))
print("Total flow:", int(total_flow))
print("Smallest outflow:", int(tot_outflows.min()), "| largest:", int(tot_outflows.max()))
print("Saved:", os.path.relpath(OD_CSV))

In [ ]:
# Local config
TILE_ID = "tile_ID"
OUTFLOW_COL = "tot_outflows"
FIT_INCLUDE_ZERO_PAIRS = False        # False: fit only on pairs present in the data
TESSELLATION_CSV = CHECKPOINT_DIR / "tessellation_outflows.csv"

# Tessellation: one row per region (id, geometry, population)
tessellation = pl[[COL_ID, "geometry", COL_POP]].rename(columns={COL_ID: TILE_ID})
tessellation = gpd.GeoDataFrame(tessellation, geometry="geometry", crs=CRS_GEO)

# Outflow of each region to other regions (row sums of the matrix, diagonal is empty)
tessellation[OUTFLOW_COL] = tot_outflows.loc[tessellation[TILE_ID]].to_numpy()

# Flows in long format; axis names set explicitly because rows and columns share the index names
flows_long = (
    od
    .rename_axis(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST)
    .stack()
    .rename(COL_FLOW)
    .reset_index()
)
flows_fit = flows_long[flows_long[COL_FLOW_ORIGIN] != flows_long[COL_FLOW_DEST]]
if not FIT_INCLUDE_ZERO_PAIRS:
    flows_fit = flows_fit[flows_fit[COL_FLOW] > 0]

fdf = skmob.FlowDataFrame(
    flows_fit,
    origin=COL_FLOW_ORIGIN,
    destination=COL_FLOW_DEST,
    flow=COL_FLOW,
    tessellation=tessellation,
    tile_id=TILE_ID,
)

tessellation[[TILE_ID, COL_POP, OUTFLOW_COL]].to_csv(TESSELLATION_CSV, index=False)

print("Tiles in tessellation:", len(tessellation))
print("Flow rows used for the fit:", len(fdf),
      "| all off-diagonal pairs:", n_regions * (n_regions - 1))
print("Total commuters in fdf:", int(fdf[COL_FLOW].sum()), "| OD matrix:", int(od.to_numpy().sum()))
print("Total outflows in tessellation:", int(tessellation[OUTFLOW_COL].sum()))
print("Saved:", os.path.relpath(TESSELLATION_CSV))

In [ ]:
# Local config
GRAVITY_TYPE = "singly constrained"
GRAVITY_VARIANTS = {"gravity_power": "power_law", "gravity_exp": "exponential"}
GRAVITY_PARAMS_CSV = CHECKPOINT_DIR / "gravity_params.csv"

gravity_models = {}
param_rows = []

for name, deterrence in GRAVITY_VARIANTS.items():
    # Record warnings instead of silencing them; only the class names are kept
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = Gravity(gravity_type=GRAVITY_TYPE, deterrence_func_type=deterrence)
        model.fit(fdf, relevance_column=COL_POP)

    gravity_models[name] = model
    param_rows.append({
        "model": name,
        "deterrence_func_type": model.deterrence_func_type,
        "gamma": model.deterrence_func_args[0],
        "origin_exp": model.origin_exp,
        "destination_exp": model.destination_exp,
        "gravity_type": model.gravity_type,
        "n_fit_pairs": len(fdf),
        "fit_warnings": ";".join(sorted({w.category.__name__ for w in caught})),
    })

gravity_params = pd.DataFrame(param_rows)
gravity_params.to_csv(GRAVITY_PARAMS_CSV, index=False)

print(gravity_params.to_string(index=False))
print("Saved:", os.path.relpath(GRAVITY_PARAMS_CSV))

In [ ]:
# Local config
MODEL_OBJECTS = {**gravity_models, "radiation": Radiation()}
REALIZATION_PATH = lambda name: CHECKPOINT_DIR / f"realizations_{name}.npy"

realizations = {}

for name in MODELS:
    path = REALIZATION_PATH(name)

    if path.exists():
        realizations[name] = np.load(path)
        print(name, "- loaded from checkpoint:", os.path.relpath(path))
        continue

    model = MODEL_OBJECTS[name]
    mats = []
    warning_types = set()
    for i in range(N_REPS):
        np.random.seed(SEED_BASE + i)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            synth = model.generate(
                tessellation,
                tile_id_column=TILE_ID,
                tot_outflows_column=OUTFLOW_COL,
                relevance_column=COL_POP,
                out_format="flows",
            )
        warning_types |= {w.category.__name__ for w in caught}

        # FlowDataFrame -> plain DataFrame -> matrix in the region order of pl
        mat = (
            pd.DataFrame(synth)
            .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
            .reindex(index=ids, columns=ids)
            .fillna(0)
            .to_numpy()
        )
        mats.append(mat)

    realizations[name] = np.stack(mats)          # shape: (N_REPS, n_regions, n_regions)
    np.save(path, realizations[name])
    print(name, "- generated and saved:", os.path.relpath(path),
          "| warnings:", ";".join(sorted(warning_types)) or "none")

print()
for name in MODELS:
    arr = realizations[name]
    diag_sum = float(np.trace(arr, axis1=1, axis2=2).sum())
    mean_total = float(arr.sum(axis=(1, 2)).mean())
    print(f"{name}: shape {arr.shape} | diagonal sum {diag_sum:.0f} | "
          f"mean total {mean_total:.0f} (observed {observed_offdiag.sum():.0f})")

In [ ]:
# Local config
METRICS_CSV = CHECKPOINT_DIR / "metrics_per_realization.csv"
SUMMARY_CSV = CHECKPOINT_DIR / "metrics_summary.csv"


def cpc(observed, predicted):
    """Common Part of Commuters: 2 * sum(min) / (sum(obs) + sum(pred))."""
    return 2 * np.minimum(observed, predicted).sum() / (observed.sum() + predicted.sum())


def r2_positive(observed, predicted):
    """R2 on the raw scale, only on pairs with observed flow > 0."""
    mask = observed > 0
    return r2_score(observed[mask], predicted[mask])


# Region indices without the capital (position in the id order of pl)
keep = np.array([i for i, c in enumerate(ids) if c != CAPITAL_ID])
no_cap = np.ix_(keep, keep)
assert len(keep) == n_regions - 1

rows = []
for name in MODELS:
    for rep, pred in enumerate(realizations[name]):
        rows.append({
            "model": name,
            "rep": rep,
            "cpc": cpc(observed_offdiag, pred),
            "r2": r2_positive(observed_offdiag, pred),
            "cpc_no_capital": cpc(observed_offdiag[no_cap], pred[no_cap]),
        })

metrics = pd.DataFrame(rows)
metrics.to_csv(METRICS_CSV, index=False)

metric_cols = ["cpc", "r2", "cpc_no_capital"]
summary = metrics.groupby("model", sort=False)[metric_cols].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
summary.to_csv(SUMMARY_CSV, index=False)

print(summary.round(4).to_string(index=False))
print("Saved:", os.path.relpath(METRICS_CSV), "and", os.path.relpath(SUMMARY_CSV))

In [ ]:
import powerlaw

# Local config
POWERLAW_CSV = CHECKPOINT_DIR / "powerlaw_distance_test.csv"

# One data point per commuter: distance of the pair, repeated by the flow
pair_mask = ~np.eye(n_regions, dtype=bool)
pair_dist = dist_matrix_km[pair_mask]
pair_flow = observed_offdiag[pair_mask].astype(int)
distances_weighted = np.repeat(pair_dist, pair_flow)

# Fit and compare; warnings are recorded (class names only) so that no file path is printed
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fit = powerlaw.Fit(distances_weighted, discrete=False)
    R, p = fit.distribution_compare("power_law", "exponential")

result = pd.DataFrame([{
    "n_points": len(distances_weighted),
    "n_distinct_distances": int(np.unique(pair_dist[pair_flow > 0].round(6)).size),
    "xmin": fit.power_law.xmin,
    "power_law_alpha": fit.power_law.alpha,
    "R_power_vs_exp": R,
    "p": p,
    "warnings": ";".join(sorted({w.category.__name__ for w in caught})) or "none",
}])
result.to_csv(POWERLAW_CSV, index=False)

print(result.round(4).T.to_string(header=False))
print("Saved:", os.path.relpath(POWERLAW_CSV))

In [ ]:
# Local config
SCATTER_FILE = "pl_observed_vs_predicted.png"
FIG_DPI = 150
LABEL_X, LABEL_Y = "Megfigyelt flow", "Becsült flow"

obs_flat = observed_offdiag.flatten()

fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 5), sharex=True, sharey=True)
excluded = {}

for ax, name in zip(axes, MODELS):
    pred_flat = realizations[name].mean(axis=0).flatten()   # mean of the realizations
    on_observed = obs_flat > 0
    plotted = on_observed & (pred_flat > 0)
    excluded[name] = int(on_observed.sum() - plotted.sum())

    ax.scatter(obs_flat[plotted], pred_flat[plotted], alpha=0.6, s=20)
    lims = [min(obs_flat[plotted].min(), pred_flat[plotted].min()),
            max(obs_flat[plotted].max(), pred_flat[plotted].max())]
    ax.plot(lims, lims, "k--", linewidth=1)       # y = x reference line
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(LABEL_X)
    ax.set_title(name)

axes[0].set_ylabel(LABEL_Y)
plt.tight_layout()

for fig_dir in FIG_DIRS:
    out_path = fig_dir / SCATTER_FILE
    fig.savefig(out_path, dpi=FIG_DPI)
    print("Saved:", os.path.relpath(out_path))
plt.show()

print("Observed pairs plotted (flow > 0):", int((obs_flat > 0).sum()))
print("Observed pairs left out (predicted flow = 0):", excluded)

In [ ]:
import matplotlib.lines as mlines
import matplotlib.patheffects as pe
from matplotlib.collections import LineCollection

# Local config
MAP_FILE_NAMES = "pl_commuting_network.png"
MAP_FILE_CODES = "pl_commuting_network_codes.png"
EDGE_WIDTH_MIN, EDGE_WIDTH_MAX = 0.3, 6.0      # line width range (points)
EDGE_ALPHA, EDGE_COLOR = 0.35, "steelblue"
NODE_SIZE_DIVISOR = 3500                       # marker area = population / divisor (same as the CZ figure)
NODE_COLOR = "darkred"
LABEL_TOP_N = 15                               # name figure: label only the N most populous regions
LABEL_FONTSIZE_NAMES, LABEL_FONTSIZE_CODES = 9, 6
LABEL_OFFSET_BASE_M, LABEL_OFFSET_PER_SQRT_M = 10000, 500   # label shift from node (metres)
# Label side per region id: "above" (default), "below", "left", "right"
LABEL_PLACEMENT = {
    "PL911": "below",    # Miasto Warszawa
    "PL913": "left",     # Warszawski zachodni
    "PL912": "right",    # Warszawski wschodni
    "PL213": "below",    # Miasto Kraków
    "PL214": "above",    # Krakowski
}
# anchor of the text for each side: (x sign, y sign, horizontal alignment, vertical alignment)
PLACEMENT_STYLE = {
    "above": (0, 1, "center", "bottom"),
    "below": (0, -1, "center", "top"),
    "left": (-1, 0, "right", "center"),
    "right": (1, 0, "left", "center"),
}
MAP_TITLE = "PL NUTS3 ingázási hálózat (valódi flow, 2021)"

max_flow = observed_offdiag.max()
node_sizes = pl[COL_POP].to_numpy() / NODE_SIZE_DIVISOR
top_ids = set(pl.nlargest(LABEL_TOP_N, COL_POP)[COL_ID])


def draw_flow_map(label_by_id, file_name, title, fontsize):
    """Draw the observed flow network; label_by_id maps region id -> label text."""
    fig, ax = plt.subplots(figsize=(12, 10))
    pl_metric.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.8)

    # Edges as one collection, width scaled by the square root of the flow
    src, dst = np.nonzero(observed_offdiag)
    flows = observed_offdiag[src, dst]
    order = np.argsort(flows)                  # thin edges first, thick edges on top
    src, dst, flows = src[order], dst[order], flows[order]
    segments = [[coords[i], coords[j]] for i, j in zip(src, dst)]
    widths = EDGE_WIDTH_MIN + (EDGE_WIDTH_MAX - EDGE_WIDTH_MIN) * np.sqrt(flows / max_flow)
    ax.add_collection(LineCollection(segments, linewidths=widths, colors=EDGE_COLOR,
                                     alpha=EDGE_ALPHA, zorder=1, capstyle="round"))

    # Nodes: area scaled by population
    ax.scatter(coords[:, 0], coords[:, 1], s=node_sizes, color=NODE_COLOR,
               zorder=5, edgecolor="white", linewidth=0.8)

    # Labels (only for the ids present in label_by_id), placed by LABEL_PLACEMENT
    for i, region_id in enumerate(ids):
        if region_id not in label_by_id:
            continue
        shift = np.sqrt(node_sizes[i]) * LABEL_OFFSET_PER_SQRT_M + LABEL_OFFSET_BASE_M
        sx, sy, ha, va = PLACEMENT_STYLE[LABEL_PLACEMENT.get(region_id, "above")]
        ax.annotate(label_by_id[region_id],
                    (coords[i, 0] + sx * shift, coords[i, 1] + sy * shift),
                    fontsize=fontsize, fontweight="bold", ha=ha, va=va, zorder=6,
                    path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])

    ax.set_title(title, fontsize=13)
    ax.set_axis_off()
    ax.legend(handles=[
        mlines.Line2D([], [], color=EDGE_COLOR, linewidth=6, alpha=0.6,
                      label="Él vastagsága ∝ √(ingázók száma)"),
        mlines.Line2D([], [], marker="o", linestyle="", color=NODE_COLOR, markersize=10,
                      label="Csomópont mérete ∝ népesség"),
    ], loc="lower left", fontsize=10, frameon=True, framealpha=0.9)

    plt.tight_layout()
    for fig_dir in FIG_DIRS:
        out_path = fig_dir / file_name
        fig.savefig(out_path, dpi=FIG_DPI)
        print("Saved:", os.path.relpath(out_path))
    plt.show()
    return len(flows)


# Figure 1: region names, only for the most populous regions
n_edges = draw_flow_map({c: code_to_name[c] for c in top_ids}, MAP_FILE_NAMES, MAP_TITLE,
                        LABEL_FONTSIZE_NAMES)
print("Edges drawn:", n_edges)
print("Largest flow:", int(max_flow))

In [ ]:
# Figure 2: NUTS3 codes for all regions
n_edges_codes = draw_flow_map({c: c for c in ids}, MAP_FILE_CODES, MAP_TITLE + " – NUTS3 kódok",
                              LABEL_FONTSIZE_CODES)
print("Edges drawn:", n_edges_codes)

In [ ]:
# Local config: attribute names
ATTR_POP, ATTR_WEIGHT = "population", "weight"
mean_pred = {name: realizations[name].mean(axis=0) for name in MODELS}

G_pl = nx.DiGraph()
for i, node_id in enumerate(ids):
    G_pl.add_node(node_id, **{ATTR_POP: int(pl[COL_POP].iloc[i])})

src, dst = np.nonzero(observed_offdiag)
for i, j in zip(src, dst):
    attrs = {ATTR_WEIGHT: float(observed_offdiag[i, j])}
    for name in MODELS:
        attrs[f"predicted_{name}"] = float(mean_pred[name][i, j])
    G_pl.add_edge(ids[i], ids[j], **attrs)

nx.write_graphml(G_pl, GRAPH_PATH)

# Read back and check
G_check = nx.read_graphml(GRAPH_PATH)
assert G_check.number_of_nodes() == G_pl.number_of_nodes()
assert G_check.number_of_edges() == G_pl.number_of_edges()
total_weight = sum(d[ATTR_WEIGHT] for _, _, d in G_check.edges(data=True))

print("Nodes:", G_pl.number_of_nodes(), "| edges:", G_pl.number_of_edges())
print("Total edge weight:", int(total_weight), "| observed total:", int(observed_offdiag.sum()))
print("Edge attributes:", sorted(next(iter(G_check.edges(data=True)))[2].keys()))
print("Saved:", os.path.relpath(GRAPH_PATH))

In [ ]:
# Local config
CZ_PARAMS_CSV = config.PROCESSED_DIR / "05_gravity_cz" / "gravity_params.csv"
COMPARISON_METRICS_CSV = CHECKPOINT_DIR / "cz_pl_metrics_comparison.csv"
COMPARISON_PARAMS_CSV = CHECKPOINT_DIR / "cz_pl_params_comparison.csv"
METRIC_COLS = ["cpc_mean", "r2_mean", "cpc_no_capital_mean"]
PARAM_COLS = ["gamma", "destination_exp"]

# The Czech tables come from notebook 05; stop with a clear message if they are missing
for p in [CZ_SUMMARY_CSV, CZ_PARAMS_CSV]:
    assert p.exists(), "missing file (run notebook 05 first): " + os.path.relpath(p)

cz_summary = pd.read_csv(CZ_SUMMARY_CSV).set_index("model")
cz_params = pd.read_csv(CZ_PARAMS_CSV).set_index("model")
pl_summary = summary.set_index("model")
pl_params = gravity_params.set_index("model")


def side_by_side(cz_tbl, pl_tbl, cols, rows):
    """Long table: one row per (model, quantity), columns CZ, PL and the difference."""
    out = []
    for model in rows:
        for col in cols:
            if model in cz_tbl.index and model in pl_tbl.index:
                cz_v, pl_v = cz_tbl.loc[model, col], pl_tbl.loc[model, col]
                out.append({"model": model, "quantity": col,
                            "CZ": cz_v, "PL": pl_v, "PL_minus_CZ": pl_v - cz_v})
    return pd.DataFrame(out)


metrics_cmp = side_by_side(cz_summary, pl_summary, METRIC_COLS, MODELS)
params_cmp = side_by_side(cz_params, pl_params, PARAM_COLS, list(GRAVITY_VARIANTS))

metrics_cmp.to_csv(COMPARISON_METRICS_CSV, index=False)
params_cmp.to_csv(COMPARISON_PARAMS_CSV, index=False)

print("Metrics (mean over the realizations):")
print(metrics_cmp.round(4).to_string(index=False))
print()
print("Gravity parameters:")
print(params_cmp.round(4).to_string(index=False))
print()
print("Saved:", os.path.relpath(COMPARISON_METRICS_CSV), "and", os.path.relpath(COMPARISON_PARAMS_CSV))

In [ ]:
# Local config
SENS_TAG = "zeros"                      # suffix for all files of this sensitivity run
SENS_PARAMS_CSV = CHECKPOINT_DIR / f"gravity_params_{SENS_TAG}.csv"

# All off-diagonal pairs; pairs missing from the data have flow 0
flows_zero = flows_long[flows_long[COL_FLOW_ORIGIN] != flows_long[COL_FLOW_DEST]]
assert len(flows_zero) == n_regions * (n_regions - 1)
n_zero_pairs = int((flows_zero[COL_FLOW] == 0).sum())

fdf_zero = skmob.FlowDataFrame(
    flows_zero,
    origin=COL_FLOW_ORIGIN,
    destination=COL_FLOW_DEST,
    flow=COL_FLOW,
    tessellation=tessellation,
    tile_id=TILE_ID,
)

# Fit each model separately; a failed fit is recorded (class name only), not raised
gravity_models_zero = {}
rows = []
for name, deterrence in GRAVITY_VARIANTS.items():
    row = {"model": name, "n_fit_pairs": len(fdf_zero), "status": "ok", "error_class": "",
           "gamma_zeros": np.nan, "dest_exp_zeros": np.nan}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            model = Gravity(gravity_type=GRAVITY_TYPE, deterrence_func_type=deterrence)
            model.fit(fdf_zero, relevance_column=COL_POP)
            gravity_models_zero[name] = model
            row["gamma_zeros"] = model.deterrence_func_args[0]
            row["dest_exp_zeros"] = model.destination_exp
        except Exception as e:
            row["status"] = "failed"
            row["error_class"] = type(e).__name__
    row["fit_warnings"] = ";".join(sorted({w.category.__name__ for w in caught}))
    rows.append(row)

sens_params = pd.DataFrame(rows)
sens_params.to_csv(SENS_PARAMS_CSV, index=False)

# Side by side with the main run (pairs present in the data only); column names are all distinct
main_params = gravity_params[["model", "gamma", "destination_exp"]].rename(
    columns={"gamma": "gamma_main", "destination_exp": "dest_exp_main"})
cmp_params = sens_params.merge(main_params, on="model")
cmp_params["gamma_diff"] = cmp_params["gamma_zeros"] - cmp_params["gamma_main"]
cmp_params["dest_exp_diff"] = cmp_params["dest_exp_zeros"] - cmp_params["dest_exp_main"]

print("Pairs used for the fit:", len(flows_zero), "| of which zero-flow:", n_zero_pairs)
print(cmp_params[["model", "status", "error_class", "gamma_main", "gamma_zeros", "gamma_diff",
                  "dest_exp_main", "dest_exp_zeros", "dest_exp_diff"]].round(4).to_string(index=False))
print("Saved:", os.path.relpath(SENS_PARAMS_CSV))

In [ ]:
# Local config
SENS_REALIZATION_PATH = lambda name: CHECKPOINT_DIR / f"realizations_{name}_{SENS_TAG}.npy"
SENS_METRICS_CSV = CHECKPOINT_DIR / f"metrics_per_realization_{SENS_TAG}.csv"
SENS_COMPARISON_CSV = CHECKPOINT_DIR / f"metrics_comparison_{SENS_TAG}.csv"
SENS_METRIC_COLS = ["cpc", "r2", "cpc_no_capital"]

ok_models = list(gravity_models_zero)          # only models whose fit succeeded
sens_realizations = {}

for name in ok_models:
    path = SENS_REALIZATION_PATH(name)
    if path.exists():
        sens_realizations[name] = np.load(path)
        print(name, "- loaded from checkpoint:", os.path.relpath(path))
        continue

    mats = []
    for i in range(N_REPS):
        np.random.seed(SEED_BASE + i)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            synth = gravity_models_zero[name].generate(
                tessellation,
                tile_id_column=TILE_ID,
                tot_outflows_column=OUTFLOW_COL,
                relevance_column=COL_POP,
                out_format="flows",
            )
        mats.append(
            pd.DataFrame(synth)
            .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
            .reindex(index=ids, columns=ids)
            .fillna(0)
            .to_numpy()
        )
    sens_realizations[name] = np.stack(mats)
    np.save(path, sens_realizations[name])
    print(name, "- generated and saved:", os.path.relpath(path))

# Metrics per realization (same functions and capital exclusion as in section 6)
rows = []
for name in ok_models:
    for rep, pred in enumerate(sens_realizations[name]):
        rows.append({
            "model": name,
            "rep": rep,
            "cpc": cpc(observed_offdiag, pred),
            "r2": r2_positive(observed_offdiag, pred),
            "cpc_no_capital": cpc(observed_offdiag[no_cap], pred[no_cap]),
        })
sens_metrics = pd.DataFrame(rows)
sens_metrics.to_csv(SENS_METRICS_CSV, index=False)

# Side by side with the main run
sens_means = sens_metrics.groupby("model", sort=False)[SENS_METRIC_COLS].mean()
main_means = metrics[metrics["model"].isin(ok_models)].groupby("model", sort=False)[SENS_METRIC_COLS].mean()

comparison = []
for name in ok_models:
    for col in SENS_METRIC_COLS:
        comparison.append({"model": name, "quantity": col,
                           "main": main_means.loc[name, col],
                           "zeros": sens_means.loc[name, col],
                           "zeros_minus_main": sens_means.loc[name, col] - main_means.loc[name, col]})
comparison = pd.DataFrame(comparison)
comparison.to_csv(SENS_COMPARISON_CSV, index=False)

for name in ok_models:
    arr = sens_realizations[name]
    print(f"{name}: shape {arr.shape} | diagonal sum {float(np.trace(arr, axis1=1, axis2=2).sum()):.0f} | "
          f"mean total {float(arr.sum(axis=(1, 2)).mean()):.0f} (observed {observed_offdiag.sum():.0f})")
print()
print(comparison.round(4).to_string(index=False))
print("Saved:", os.path.relpath(SENS_METRICS_CSV), "and", os.path.relpath(SENS_COMPARISON_CSV))